In [ ]:
import sys
!{sys.executable} -m pip install nbformat>=4.2.0 ipywidgets scikit-learn

# EXP_009d1: Attractor Dominance, Basin Mapping & Pathway Analysis

## Depends On
**EXP_009d0 must PASS before running this notebook.**

## Hypotheses Under Test

### H1: Attractor Dominance
The `prolet` attractor is the dominant basin of GPT-2 Small's weight geometry. It should capture the majority of a diverse prompt set, regardless of input register, topic, or complexity.

### H2: Secondary Basin Existence
The `Divine` attractor is a genuine secondary basin, not a one-off artefact of a single prompt. Other prompts with similar syntactic properties should also route to `Divine` (or to other previously unseen basins).

### H3: Dissolution Pathway Structure
The intermediate tokens observed during dissolution (e.g., `Femminus Fem`) reflect the statistical topology of the training corpus, not random noise. Different input types may trace different but internally coherent pathways to the same terminal attractor.

## Predictions

| Prompt | Predicted Basin | Rationale |
|:---|:---|:---|
| Academic | `prolet` | Complex, multi-syllabic, scientific |
| Emotional | `prolet` | Personal register, complex syntax |
| Technical | `prolet` | Jargon, programming register |
| Historical | `prolet` | Narrative factual |
| Philosophical | `prolet` | Abstract reasoning |
| Journalistic | `prolet` | Media register |
| Poetic_Complex | `prolet` | Multi-syllabic literary |
| Nursery | `Divine` (?) | Simple, monosyllabic, fairy-tale |
| Fable | `Divine` (?) | Simple declarative, animal subjects |
| Scriptural | `Divine` (?) | Simple declarative, biblical syntax |
| Primer | `Divine` (?) | Monosyllabic, basic SVO |
| Nursery2 | `Divine` (?) | Repeating pattern |

**Note:** The `Divine` predictions are explicitly weaker — marked with `(?)`. We observed a single instance and are testing whether it generalises. Finding that it does not would itself be informative.

---


In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")
print(f"Architecture: {model.cfg.n_layers} layers, {model.cfg.n_heads} heads, d_model={model.cfg.d_model}")

In [ ]:
# ============================================================
# STEP 2: CONFIGURATION — Hypothesis-Driven Prompt Set
# ============================================================

# Tightened schedule: convergence occurs by ~100, no need for 250/500
# unless we observe late-converging prompts
ITERATION_SCHEDULE = [0, 2, 3, 5, 10, 20, 50, 100]
MAX_ITERATIONS = max(ITERATION_SCHEDULE)

# --- Predicted → prolet basin ---
PROLET_PROMPTS = {
    "Academic":      "The implications of quantum entanglement suggest that",
    "Emotional":     "I have never felt so alone in my entire",
    "Technical":     "The function returns a pointer to the allocated",
    "Historical":    "Napoleon crossed the Alps with an army of",
    "Philosophical": "The categorical imperative demands that we treat each",
    "Journalistic":  "According to sources familiar with the matter the",
    "Poetic_Complex": "Through the labyrinthine corridors of forgotten memory the",
}

# --- Predicted → Divine basin (weak prediction) ---
DIVINE_PROMPTS = {
    "Nursery":    "Jack and Jill went up the hill to",
    "Fable":      "The fox and the hen sat by the",
    "Scriptural": "And God said let there be light and",
    "Primer":     "The dog ran to the big red box",
    "Nursery2":   "Old King Cole was a merry old soul",
}

# Combined library
PROMPT_LIBRARY = {}
PROMPT_LIBRARY.update(PROLET_PROMPTS)
PROMPT_LIBRARY.update(DIVINE_PROMPTS)

LAYER_START = 0
LAYER_END = model.cfg.n_layers - 1

print(f"Schedule: {ITERATION_SCHEDULE}")
print(f"Room: Layers {LAYER_START} → {LAYER_END}")
print(f"\nPredicted → prolet ({len(PROLET_PROMPTS)} prompts):")
for k, v in PROLET_PROMPTS.items():
    print(f"  {k}: \"{v}\"")
print(f"\nPredicted → Divine (weak, {len(DIVINE_PROMPTS)} prompts):")
for k, v in DIVINE_PROMPTS.items():
    print(f"  {k}: \"{v}\"")

In [ ]:
# ============================================================
# STEP 3: THE CORE ENGINE — Identical to EXP_009aFIX/d0
# ============================================================

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions."""
    normalized = model.ln_final(resid_vector)
    logits = normalized @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.tolist()))


def run_total_resonance_loop(model, prompt, layer_start, layer_end, max_iter, schedule):
    """Total Lucier Loop with norm normalisation and full-position decoding."""
    snapshots = []
    hook_point_read = f"blocks.{layer_end}.hook_resid_post"
    hook_point_write = f"blocks.{layer_start}.hook_resid_pre"
    
    with torch.no_grad():
        _, cache = model.run_with_cache(
            prompt,
            names_filter=lambda n: n == hook_point_read
        )
    
    current_tensor = cache[hook_point_read][0].clone()
    seq_len = current_tensor.shape[0]
    initial_norm = current_tensor.norm().item()
    print(f"  Seq len: {seq_len}, initial norm: {initial_norm:.2f}")
    
    last_vec = current_tensor[-1, :].clone()
    mean_vec = current_tensor.mean(dim=0).clone()
    
    if 0 in schedule:
        top_tokens_last = get_top_tokens(model, last_vec)
        all_pos_tokens = []
        for pos in range(seq_len):
            pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
            all_pos_tokens.append(pos_top[0][0])
        snapshots.append({
            "iteration": 0,
            "tensor": current_tensor.clone().cpu(),
            "last_vector": last_vec.clone().cpu(),
            "mean_vector": mean_vec.clone().cpu(),
            "last_norm": last_vec.norm().item(),
            "mean_norm": mean_vec.norm().item(),
            "tensor_norm": current_tensor.norm().item(),
            "top_tokens": top_tokens_last,
            "all_position_tokens": all_pos_tokens,
            "cosine_sim_last": 1.0,
            "cosine_sim_mean": 1.0,
            "position_similarity": 1.0,
        })
    
    prev_last = last_vec.clone()
    prev_mean = mean_vec.clone()
    
    for i in range(1, max_iter + 1):
        current_norm = current_tensor.norm().item()
        if current_norm > 0:
            current_tensor = current_tensor * (initial_norm / current_norm)
        
        inject_tensor = current_tensor.clone()
        
        def injection_hook(resid, hook, tensor=inject_tensor):
            resid[0, :, :] = tensor
            return resid
        
        model.add_hook(hook_point_write, injection_hook)
        try:
            with torch.no_grad():
                _, cache = model.run_with_cache(
                    prompt,
                    names_filter=lambda n: n == hook_point_read
                )
        finally:
            model.reset_hooks()
        
        current_tensor = cache[hook_point_read][0].clone()
        last_vec = current_tensor[-1, :].clone()
        mean_vec = current_tensor.mean(dim=0).clone()
        
        if i in schedule:
            cos_sim_last = torch.nn.functional.cosine_similarity(
                last_vec.unsqueeze(0), prev_last.unsqueeze(0)
            ).item()
            cos_sim_mean = torch.nn.functional.cosine_similarity(
                mean_vec.unsqueeze(0), prev_mean.unsqueeze(0)
            ).item()
            
            pos_norms = current_tensor.norm(dim=1, keepdim=True).clamp(min=1e-8)
            normalized_positions = current_tensor / pos_norms
            pos_sim_matrix = normalized_positions @ normalized_positions.T
            mask = ~torch.eye(seq_len, dtype=torch.bool, device=pos_sim_matrix.device)
            position_similarity = pos_sim_matrix[mask].mean().item()
            
            top_tokens_last = get_top_tokens(model, last_vec)
            all_pos_tokens = []
            for pos in range(seq_len):
                pos_top = get_top_tokens(model, current_tensor[pos, :], k=1)
                all_pos_tokens.append(pos_top[0][0])
            
            snapshots.append({
                "iteration": i,
                "tensor": current_tensor.clone().cpu(),
                "last_vector": last_vec.clone().cpu(),
                "mean_vector": mean_vec.clone().cpu(),
                "last_norm": last_vec.norm().item(),
                "mean_norm": mean_vec.norm().item(),
                "tensor_norm": current_tensor.norm().item(),
                "top_tokens": top_tokens_last,
                "all_position_tokens": all_pos_tokens,
                "cosine_sim_last": cos_sim_last,
                "cosine_sim_mean": cos_sim_mean,
                "position_similarity": position_similarity,
            })
            print(f"  iter {i:>3}: top='{top_tokens_last[0][0].strip()}', "
                  f"cos_mean={cos_sim_mean:.4f}, pos_collapse={position_similarity:.4f}")
        
        prev_last = last_vec.clone()
        prev_mean = mean_vec.clone()
    
    return snapshots

print("Engine loaded.")

In [ ]:
# ============================================================
# STEP 4: RUN ALL PROMPTS
# ============================================================

all_results = {}

for label, prompt in PROMPT_LIBRARY.items():
    print(f"\n{'='*60}")
    print(f"RECORDING: '{label}' — \"{prompt}\"")
    print(f"{'='*60}")
    
    snapshots = run_total_resonance_loop(
        model, prompt,
        layer_start=LAYER_START,
        layer_end=LAYER_END,
        max_iter=MAX_ITERATIONS,
        schedule=ITERATION_SCHEDULE
    )
    all_results[label] = snapshots
    print(f"  ✓ {len(snapshots)} snapshots captured.")

print(f"\n{'='*60}")
print(f"ALL RECORDINGS COMPLETE.")

---
## 5. Analysis

### 5a. Sentence Dissolution Tables — Full Position Reconstruction

In [ ]:
# ============================================================
# VIS 5a: SENTENCE DISSOLUTION TABLES
# ============================================================

for label in PROMPT_LIBRARY.keys():
    snapshots = all_results[label]
    predicted = "prolet" if label in PROLET_PROMPTS else "Divine (?)"
    md = f"### {label} (predicted → `{predicted}`): *\"{PROMPT_LIBRARY[label]}\"*\n\n"
    md += "| Iter | Reconstructed Output |\n"
    md += "|:---|:---|\n"
    for s in snapshots:
        tokens = s['all_position_tokens']
        clean = [t.replace('\n', '↵').replace('|', '\\|') for t in tokens]
        sentence = ' '.join(clean)
        md += f"| {s['iteration']} | {sentence} |\n"
    md += "\n"
    display(Markdown(md))

### 5b. Hypothesis Assessment — Where Did Each Prompt Land?

In [ ]:
# ============================================================
# VIS 5b: HYPOTHESIS ASSESSMENT — Predictions vs Actuals
# ============================================================

md = "## Prediction Results\n\n"
md += "| Prompt | Predicted Basin | Actual Terminal Token | Match? |\n"
md += "|:---|:---|:---|:---|\n"

prolet_count = 0
divine_count = 0
other_count = 0
other_tokens = []

for label, snapshots in all_results.items():
    terminal = snapshots[-1]['top_tokens'][0][0].strip()
    predicted = "prolet" if label in PROLET_PROMPTS else "Divine"
    
    if 'prolet' in terminal or terminal in 'prolet':
        actual_basin = "prolet"
        prolet_count += 1
    elif 'Divine' in terminal or terminal in 'Divine':
        actual_basin = "Divine"
        divine_count += 1
    else:
        actual_basin = f"OTHER: {terminal}"
        other_count += 1
        other_tokens.append((label, terminal))
    
    match = "✓" if predicted.lower() in actual_basin.lower() else "✗"
    md += f"| {label} | `{predicted}` | `{terminal}` ({actual_basin}) | {match} |\n"

md += "\n"
md += f"**Summary:** {prolet_count} → prolet, {divine_count} → Divine, {other_count} → other basins\n"

if other_tokens:
    md += "\n**New basins discovered:**\n"
    for label, tok in other_tokens:
        md += f"- `{label}` → `{tok}` (previously unseen attractor)\n"

display(Markdown(md))

### 5c. Cross-Prompt Convergence Matrix

In [ ]:
# ============================================================
# VIS 5c: CROSS-PROMPT CONVERGENCE MATRIX
# ============================================================

labels = list(all_results.keys())
n = len(labels)
sim_matrix = np.zeros((n, n))

final_vectors = []
for label in labels:
    final_vec = all_results[label][-1]["mean_vector"]
    final_vectors.append(final_vec)

for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = torch.nn.functional.cosine_similarity(
            final_vectors[i].unsqueeze(0).float(),
            final_vectors[j].unsqueeze(0).float()
        ).item()

fig_sim = px.imshow(
    sim_matrix,
    x=labels, y=labels,
    color_continuous_scale="Viridis",
    title="EXP_009d1: Cross-Prompt Convergence Matrix (12 Prompts)",
    text_auto=".3f",
    aspect="auto",
)
fig_sim.update_layout(template="plotly_dark", height=600)
fig_sim.show()

### 5d. Dissolution Pathway Analysis (Stage 3)
For each prompt, extract the intermediate tokens at each phase to trace the discourse topology.

In [ ]:
# ============================================================
# VIS 5d: PATHWAY COMPARISON — What routes do different inputs take?
# ============================================================

md = "## Dissolution Pathways — Last-Token Top Prediction\n\n"
md += "| Iter | " + " | ".join(PROMPT_LIBRARY.keys()) + " |\n"
md += "| :--- | " + " | ".join([":---"] * len(PROMPT_LIBRARY)) + " |\n"

for idx, iteration in enumerate(ITERATION_SCHEDULE):
    row = f"| **{iteration}** |"
    for label in PROMPT_LIBRARY.keys():
        snapshots = all_results[label]
        if idx < len(snapshots):
            tok = snapshots[idx]['top_tokens'][0][0]
            clean_t = tok.replace('\n', '↵').replace('`', "'").strip()
            row += f" `{clean_t}` |"
        else:
            row += " — |"
    md += row + "\n"

display(Markdown(md))

### 5e. 3D PCA Trajectories — All 12 Prompts

In [ ]:
# ============================================================
# VIS 5e: 3D PCA TRAJECTORIES
# ============================================================
from sklearn.decomposition import PCA
import pandas as pd

all_vecs = []
labels_list = []
iters_list = []
text_list = []

for label, snapshots in all_results.items():
    for s in snapshots:
        all_vecs.append(s["mean_vector"].detach().cpu().numpy())
        labels_list.append(label)
        iters_list.append(s["iteration"])
        top_tok = s['top_tokens'][0][0].replace('\n', '↵').strip()
        text_list.append(f"Iter {s['iteration']}: {top_tok}")

all_vecs = np.array(all_vecs)
pca = PCA(n_components=3)
vecs_3d = pca.fit_transform(all_vecs)

df = pd.DataFrame({
    'x': vecs_3d[:, 0],
    'y': vecs_3d[:, 1],
    'z': vecs_3d[:, 2],
    'Prompt': labels_list,
    'Iteration': iters_list,
    'Top_Token': text_list
})

fig_topo = px.line_3d(
    df, x='x', y='y', z='z',
    color='Prompt',
    hover_name='Top_Token',
    markers=True,
    title=f"EXP_009d1: Attractor Landscape — 12 Prompt Trajectories<br>"
          f"<sup>(Explained Variance: {sum(pca.explained_variance_ratio_)*100:.1f}%)</sup>"
)
fig_topo.update_traces(marker=dict(size=4), line=dict(width=3))
fig_topo.update_layout(
    template="plotly_dark",
    height=800,
    scene=dict(
        xaxis_title="PC 1",
        yaxis_title="PC 2",
        zaxis_title="PC 3",
    )
)
fig_topo.show()

In [ ]:
# ============================================================
# STEP 6: SAVE ARTIFACTS
# ============================================================
import os

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

save_data = {}
for label, snapshots in all_results.items():
    save_data[label] = {
        "iterations": [s["iteration"] for s in snapshots],
        "last_vectors": torch.stack([s["last_vector"] for s in snapshots]),
        "mean_vectors": torch.stack([s["mean_vector"] for s in snapshots]),
        "last_norms": [s["last_norm"] for s in snapshots],
        "mean_norms": [s["mean_norm"] for s in snapshots],
        "cosine_sims_last": [s["cosine_sim_last"] for s in snapshots],
        "cosine_sims_mean": [s["cosine_sim_mean"] for s in snapshots],
        "position_similarity": [s["position_similarity"] for s in snapshots],
        "top_tokens": [s["top_tokens"] for s in snapshots],
        "all_position_tokens": [s["all_position_tokens"] for s in snapshots],
    }

torch.save(save_data, os.path.join(save_dir, "009d1_attractor_dominance_results.pt"))
print(f"[SAVED] {save_dir}/009d1_attractor_dominance_results.pt")

config = {
    "schedule": ITERATION_SCHEDULE,
    "layer_start": LAYER_START,
    "layer_end": LAYER_END,
    "prolet_prompts": PROLET_PROMPTS,
    "divine_prompts": DIVINE_PROMPTS,
    "model": "gpt2-small",
    "mode": "hypothesis_driven_validation",
}
torch.save(config, os.path.join(save_dir, "009d1_config.pt"))
print(f"[SAVED] {save_dir}/009d1_config.pt")